# Inferring Topics from IMDB Reviews

In [9]:
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import pandas as pd
import matplotlib.pyplot as plt

## Exploring the Dataset: [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz)

In [10]:
# Download the archive (approx 80MB)
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz
ROOT = './aclImdb/train/pos/'

--2026-02-26 09:46:58--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz.3’

aclImdb_v1.tar.gz.3 100%[===================>]  80.23M  4.70MB/s    in 16s     

2026-02-26 09:47:14 (5.03 MB/s) - ‘aclImdb_v1.tar.gz.3’ saved [84125825/84125825]



In [11]:
reviews = []
for file in os.listdir(ROOT):
    path = os.path.join(ROOT, file)
    if os.path.isfile(path):
        with open(path, 'r') as fin:
            reviews.append(fin.read())

In [12]:
len(reviews)

12500

In [13]:
for i in range(3):
    print(reviews[i])
    print('=' * 150)

First, let me say that Notorious is an absolutely charming film, very lovingly rendered of its time and subject(s). Gretchen Mol is utterly, painfully convincing, the very soul of the contradictions smoothly reified by Ms. Page herself. Irving and Paula Klaw are richly drawn as the working-class stiffs they were (having met Paula at Movie Star News in 1990 I can say that Lili Taylor's performance is unimpeachable), and Jared Harris as John Willie (Coutts) is an adoringly debauched genius. Anyone with an interest in the recorded history of American attitudes toward sexuality must see this movie, in a theater preferably, where votes made with dollars count more.<br /><br />Second, I will allow that I am a producer of material similar to that for which the Klaws would become famous, which is no way affects my estimation of Ms. Harron's work as the splendid piece that it is, but does condition my view of Notorious as an act of political resistance of the first order. Ms. Harron has crafted

## Feature Extraction

In [14]:
vect = TfidfVectorizer(stop_words='english')
X = vect.fit_transform(reviews)

pd.DataFrame(X.toarray(), columns=vect.get_feature_names_out())

,00,000,000s,003830,006,007,0079,0080,0083,0093638,...,élan,émigré,émigrés,était,état,étc,êxtase,ís,østbye,über
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12498,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## NMF Decomposition

In [15]:
N_TOPICS = 15
nmf = NMF(n_components=N_TOPICS)
W = nmf.fit_transform(X)  # Document-topic matrix
H = nmf.components_       # Topic-term matrix

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [16]:
# Top 10 words per topic

words = np.array(vect.get_feature_names_out())
topic_words = pd.DataFrame(np.zeros((N_TOPICS, 10)), index=[f'Topic {i + 1}' for i in range(N_TOPICS)],
                           columns=[f'Word {i + 1}' for i in range(10)]).astype(str)
for i in range(N_TOPICS):
    ix = H[i].argsort()[::-1][:10]
    topic_words.iloc[i] = words[ix]

topic_words

,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10
Topic 1,br,10,ll,spoilers,end,simply,yes,spoiler,quite,just
Topic 2,movie,movies,watch,recommend,10,seen,saw,best,definitely,actors
Topic 3,film,films,seen,director,characters,cinema,festival,work,scenes,art
Topic 4,series,episode,episodes,season,tv,characters,trek,seasons,shows,television
Topic 5,man,role,character,performance,plays,john,best,played,does,actor
Topic 6,good,pretty,bad,story,acting,really,job,liked,nice,actors
Topic 7,war,world,documentary,people,american,history,soldiers,men,women,hitler
Topic 8,funny,comedy,laugh,hilarious,eddie,jokes,fun,humor,funniest,murphy
Topic 9,like,think,really,just,don,people,know,say,didn,lot
Topic 10,time,years,saw,old,dvd,kids,remember,disney,music,seen


In [17]:
# Create a topic mapping

topic_mapping = {
    'Topic 4': 'TV',
    'Topic 7': 'War',
    'Topic 8': 'Comedy',
    'Topic 12': 'Book Adaptation',
    'Topic 13': 'Horror',
    'Topic 15': 'Martial Arts / Action'
}

In [18]:
# Recall the document-topic matrix, W

W = pd.DataFrame(W, columns=[f'Topic {i + 1}' for i in range(N_TOPICS)])
W['max_topic'] = W.apply(lambda x: topic_mapping.get(x.idxmax()), axis=1)
W[pd.notnull(W['max_topic'])].head(10)

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11,Topic 12,Topic 13,Topic 14,Topic 15,max_topic
1,0.047788,0.057852,0.000000,0.005686,0.000000,0.031489,0.000000,0.000432,0.020103,0.000867,0.000000,0.000000,0.079392,0.017521,0.000000,Horror
3,0.000000,0.007729,0.028062,0.000214,0.000000,0.006899,0.000000,0.000000,0.005333,0.021936,0.008100,0.000000,0.061887,0.004036,0.000349,Horror
4,0.047494,0.009702,0.000000,0.125228,0.000000,0.011217,0.000000,0.000000,0.000000,0.000000,0.006357,0.000000,0.004334,0.000000,0.000658,TV
16,0.000132,0.003673,0.001628,0.032002,0.000000,0.000000,0.006217,0.000174,0.018610,0.000000,0.012746,0.028518,0.002773,0.004689,0.000268,TV
22,0.043468,0.000000,0.000000,0.103271,0.000000,0.001151,0.000000,0.000000,0.011135,0.000000,0.015443,0.000000,0.003002,0.000000,0.000848,TV
25,0.020186,0.000000,0.002027,0.026432,0.000000,0.014980,0.023094,0.001906,0.022413,0.000000,0.025637,0.000000,0.000000,0.006322,0.003777,TV
28,0.040899,0.000000,0.000000,0.000000,0.046651,0.005677,0.000000,0.000000,0.004360,0.004502,0.020295,0.012118,0.068676,0.002021,0.000000,Horror
37,0.027255,0.000000,0.000688,0.087535,0.031061,0.002658,0.005203,0.000000,0.007609,0.009508,0.007979,0.000000,0.004943,0.010638,0.000000,TV
40,0.000000,0.000000,0.040631,0.000000,0.000000,0.018403,0.000408,0.015400,0.006639,0.000000,0.013618,0.043839,0.000000,0.009799,0.000000,Book Adaptation
45,0.030578,0.005335,0.008915,0.000000,0.019624,0.003988,0.000000,0.034847,0.008937,0.014653,0.002835,0.000000,0.000000,0.004440,0.000000,Comedy


In [19]:
reviews[58]

'Lindy (Meryl Streep) and her husband Michael (Sam Neill) have just welcomed a baby girl, Azaria. As Seventh Day Adventists, they live their beliefs every day and soon have Azaria dedicated to God at their church, with their two older boys looking on. Michael gets a vacation and the family decides to head to Ayer\'s Rock, one of the most impressive tourist spots in all of Australia. Not being wealthy, the family camps near the site. After a wonderful first day, Lindy puts baby Azaria to sleep in one of the tents. Suddenly, she hears Azaria crying. As Lindy rushes to the tent, a dingo dog is just exiting, shaking his head. The baby is gone and soon, so is the dingo. Although the entire camp looks for the baby, she is not found. Concluding she is dead and that the dingo made off with their beloved child, the Chamberlains struggle to accept God\'s decision and go on with their lives. But, unfortunately, the story gets sensational coverage in the news media and soon the tale is circulated 

In [20]:
# Frobenius norm

import numpy as np

print("Frobenius norm and the condition number:")
print(np.linalg.norm([[1,1,1],[3,4,1],[4,1,2]], 'fro'))
print(np.linalg.cond([[1,1,1],[3,4,1],[4,1,2]], 'fro'))


Frobenius norm and the condition number:
7.0710678118654755
13.975424859373685


In [21]:
import datetime, pytz; 
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-02-26 15:17:53


In [ ]:
# --- gcolab wrapper: list files in outputs/ ---
import os, datetime
_out_dir = 'outputs'
os.makedirs(_out_dir, exist_ok=True)
print(f"Files in {_out_dir}:")
_files = []
for _fn in os.listdir(_out_dir):
    _fp = os.path.join(_out_dir, _fn)
    if os.path.isfile(_fp):
        _st = os.stat(_fp)
        _files.append((datetime.datetime.fromtimestamp(_st.st_mtime).strftime('%Y-%m-%d %H:%M:%S'), _st.st_size, _fn))
_files.sort()
for _m, _s, _n in _files:
    if _s < 1024:
        _h = f"{_s} B"
    elif _s < 1024*1024:
        _h = f"{_s/1024:.2f} KB"
    else:
        _h = f"{_s/(1024*1024):.2f} MB"
    print(f"{_m}  {_h:>10}  {_n}")


In [ ]:
# --- gcolab wrapper: zip outputs/ -> outputs.zip ---
import os, shutil
os.makedirs('outputs', exist_ok=True)
shutil.make_archive('outputs', 'zip', 'outputs')
print('Created outputs.zip')
